# Reproducible immune PCA, imputation and figure associations

This notebook performs the numerical analysis for **Figure 2, Extended Figure 2 and Figure 3**. Starting from repository measurements, it recomputes imputation and PCA, fits the association models, and publishes prepared numerical outputs in `data/`. The figure scripts read those outputs and construct the plots; they do not rerun imputation, PCA fitting or statistical models. No saved plot object or dated output folder is an input.

**Run all cells of this notebook before generating Figure 2, Extended Figure 2 or Figure 3.** Keep its prepared `data/` files in a final checkout. Regenerate them after changes to the upstream measurements or analytical settings. Figure 1 and Figure 4 retain their existing workflows. Run instructions are in `R_updated/README.md`.

## Design and interpretation

- A PBMC record anchors each patient/timepoint; cfDNA Shannon diversity is joined by `sample_id`.
- Eleven immune features enter the model. Exclude profiles missing eight or more features, or whose observed feature sum is zero. The supplied data produce 235 eligible profiles from 84 patients; 75 profiles from 47 patients are complete cases.
- Regularized rank-2 missMDA reconstruction uses only immune measurements. Patient, treatment, cancer, age, and outcomes are not imputation/PCA predictors. Observed measurements are preserved.
- Fit one centred, standardized PCA on the completed 235-row matrix, then project the 75 complete profiles onto it. **Original and missMDA overlap**: the full completed cohort includes every Original profile. Open circles denote membership in the completed cohort, not that every measurement in that sample was imputed.
- The shared reference is descriptive and includes all timepoints. It is not a prospective outcome-prediction pipeline. Any future prediction analysis must refit imputation and PCA inside patient-level training folds.


In [1]:
# Run from the repository root or from notebooks/.
candidates <- c(getwd(), file.path(getwd(), ".."))
is_project <- vapply(candidates, function(path) {
  file.exists(file.path(path, "data", "metadata.rds")) &&
    dir.exists(file.path(path, "R_updated"))
}, logical(1))
if (!any(is_project)) stop("Open this notebook from the repository root or notebooks/.")
project_root <- normalizePath(candidates[which(is_project)[1]])

required <- c("missMDA", "lme4", "lmerTest", "emmeans")
missing <- required[!vapply(required, requireNamespace, quietly = TRUE,
                           FUN.VALUE = logical(1))]
if (length(missing)) stop("Install analysis packages listed in R_updated/README.md: ",
                          paste(missing, collapse = ", "))
source(file.path(project_root, "R_updated/functions", "pca_analysis.R"))
source(file.path(project_root, "R_updated/functions", "pca_associations.R"))
source(file.path(project_root, "R_updated/functions", "pca_figure_io.R"))
tables_dir <- file.path(project_root, "tables", "pca_analysis")
dir.create(tables_dir, recursive = TRUE, showWarnings = FALSE)


## Rebuild measurements, eligibility, imputation and shared axes

The feature order is ATC, Naïve%, SCM%, CM%, EM%, TE%, PD1%, LAG3%, TIM3%, PBMC TCR Shannon diversity and cfDNA TCR Shannon diversity. Regularized `missMDA::imputePCA()` uses `ncp=2`, `scale=TRUE`, five starts, seed 20261012, tolerance 1e-6 and maximum 1000 iterations. The PCA signs are anchored to positive TE% on PC1 and positive Naïve% on PC2; signs do not change distances or statistical evidence.

Rank 2 is held fixed to reproduce the manuscript panels. The optional following cell reruns the original masked-entry rank selection: 20 repeated 10% masks, ranks 0–8 and regularized K-fold cross-validation. This evaluates reconstruction, not out-of-patient clinical prediction.


In [2]:
inputs <- prepare_pca_inputs(project_root, ncp = 2L, seed = 20261012L)
preparation_audit <- write_pca_source_data(inputs, tables_dir)
print(preparation_audit, row.names = FALSE)
stopifnot(nrow(inputs$raw235) == 235L,
          nrow(inputs$real_features75) == 75L,
          sum(is.na(inputs$raw235)) == 247L,
          max(abs(inputs$completed235[!is.na(inputs$raw235)] -
                    inputs$raw235[!is.na(inputs$raw235)])) == 0)
print(table(inputs$scoredata310$dataset, inputs$scoredata310$cycle))


                       measure     value
              eligible_samples 235.00000
             eligible_patients  84.00000
              complete_samples  75.00000
             complete_patients  47.00000
         X01_completed_samples  72.00000
          X01_complete_samples  24.00000
                 imputed_cells 247.00000
 observed_entry_max_abs_change   0.00000
          PC1_variance_percent  28.44958
          PC2_variance_percent  20.16599
          
           X01 X02 X03 X04 X05
  missMDA   72  53  47  36  27
  Original  24  19  15  10   7


In [3]:
# Optional: computationally heavier reconstruction-rank sensitivity.
# The fixed-rank inputs above remain the source for the manuscript figures.
RUN_RANK_CV <- FALSE
if (RUN_RANK_CV) {
  cv_inputs <- prepare_pca_inputs(project_root, rerun_rank_cv = TRUE)
  print(cv_inputs$rank_cv)
  write_pca_source_data(cv_inputs, file.path(tables_dir, "rank_crossvalidation"))
}


## Refit cancer and treatment associations

Panel d uses one X01 record per patient: PC1 ~ clinical lymphoma status + age, with HC3 covariance and residual-t confidence intervals. Clinical lymphoma includes ALCL; the recorded source cancer class is retained in a separate sensitivity analysis.

Panels e–f summarize all relevant baseline and exposed samples. Their primary model is PC score ~ regimen + (1 | Patient), with REML, Satterthwaite degrees of freedom and a patient random intercept. A separate sensitivity adds age and recorded cancer class. A post-treatment Xk sample carries the regimen from cycle k−1.

Holm correction includes both baseline cohort contrasts (2 tests); all estimable treatment-versus-baseline contrasts across both PCs and cohorts (30 tests, separately for primary and adjusted models); and the four selected paired signed-rank sensitivities (4 tests). Overlap is allowed under Holm; no between-cohort significance test is performed. Confidence intervals are pointwise 95%, while P values are adjusted within the stated families.

The paired rank sensitivity selects the first available eligible treatment sample and the same patient's X01, once per regimen. Rainclouds display the full sample population; the paired sensitivity has a smaller, explicitly reported patient count. Random intercepts do not remove treatment-indication or time confounding. Single imputation does not propagate missing-data uncertainty into confidence intervals; completed-data results remain sensitivity analyses.


In [4]:
statistics <- fit_pca_associations(inputs)
write_pca_association_tables(statistics, tables_dir)
print(statistics$families, row.names = FALSE)
print(statistics$baseline_lymphoma[, c("dataset", "n", "n_lymphoma", "estimate",
                                      "lower", "upper", "p_Holm")], row.names = FALSE)
selected <- statistics$regimen_contrasts
selected <- selected[selected$analysis == "Current_patient_intercept_primary" &
  ((selected$regimen == "Vincristine/Cyclophosphamide" & selected$PC == "PC1") |
     (selected$regimen == "Vincristine" & selected$PC == "PC2")), ]
print(selected[, c("dataset", "PC", "regimen", "estimate", "lower.CL", "upper.CL",
                   "p.value", "p_Holm")], row.names = FALSE)
print(statistics$diagnostics, row.names = FALSE)
# Singular Original-PC2 fits are retained and explicitly flagged, as in the
# existing analysis; use the sensitivity tables when assessing robustness.


                           family n_tests correction
        Baseline cancer contrasts       2       Holm
        Primary regimen contrasts      30       Holm
     Adjusted regimen sensitivity      30       Holm
 Selected paired rank sensitivity       4       Holm
  dataset  n n_lymphoma  estimate     lower      upper      p_Holm
 Original 24          4 -2.302358 -3.513939 -1.0907760 0.001457729
  missMDA 72         11 -1.760444 -2.977010 -0.5438777 0.005192064
  dataset  PC                      regimen    estimate   lower.CL   upper.CL
 Original PC1 Vincristine/Cyclophosphamide -1.34998684 -2.5689374 -0.1310363
 Original PC2                  Vincristine  2.14317554  1.0345177  3.2518334
  missMDA PC1 Vincristine/Cyclophosphamide -1.16673291 -1.7913542 -0.5421116
  missMDA PC2                  Vincristine  0.05030194 -0.5039527  0.6045566
      p.value      p_Holm
 0.0304919342 0.670822552
 0.0002589499 0.007768497
 0.0002922469 0.008475161
 0.8581857636 1.000000000
  dataset  PC      

Message: boundary (singular) fit: see help('isSingular')

Message: boundary (singular) fit: see help('isSingular')



## Publish prepared numerical inputs for the figure scripts

The files below are the required hand-off from this notebook to the figure scripts:

- `data/pca_inputs.rds`: completed/observed feature matrices, sample metadata, shared PCA reference, scores and cohort information.
- `data/association_results.rds`: fitted effect estimates, confidence intervals, adjusted P values, paired sensitivities and diagnostics as numerical tables.

The bundle contains numerical data and a PCA reference, with no saved ggplot objects. Figure scripts read the bundle and construct fresh plots. They stop with instructions to run this notebook when required outputs are missing; they do not silently regenerate the analysis. Extended 2 and Figure 3 need only the PCA inputs, while Figure 2 also reads the statistical results.

All statistical choices, point positions, cohort counts and plotting styles are retained. Figure 3 constructs its trajectory edges from the saved score metadata when drawing the figure.

`pca_inputs.rds` is required by all three plotting scripts. It contains the 235-row completed-cohort PCA and the 75 Original profiles projected onto those axes, together with the feature matrices and sample metadata. The older `data/pca.rds` contains a different, Original-only PCA and cannot supply these shared-axis completed-cohort plots. No separate manifest is written or required.


In [5]:
write_pca_figure_outputs(inputs, statistics, project_root)
prepared <- read_pca_figure_outputs(project_root)
stopifnot(isTRUE(all.equal(prepared$inputs$completed235, inputs$completed235)),
          isTRUE(all.equal(prepared$inputs$reference_pca$x, inputs$reference_pca$x)),
          isTRUE(all.equal(prepared$statistics$baseline_lymphoma,
                           statistics$baseline_lymphoma)),
          isTRUE(all.equal(prepared$statistics$regimen_contrasts,
                           statistics$regimen_contrasts)))
cat("Prepared PCA inputs and statistical results saved in data/.\n")


Prepared PCA inputs and statistical results saved in data/.


## Generate the figures and keep the reproducibility record

After this notebook completes, run the following from the repository root:

```sh
Rscript R_updated/Figure2_updated.R
Rscript R_updated/Extended2_updated.R
Rscript R_updated/Figure3_updated.R
```

The scripts read the prepared numerical files and export fresh PDF/PNG/SVG figures and plotting-source tables. Analysis packages such as missMDA and lmerTest are needed to rerun this notebook, but are not needed for the three plotting scripts. A prepared checkout can therefore redraw these figures without the upstream measurement files or analysis helpers.

The notebook also exports raw/completed matrices, eligibility, scores, loadings, settings, input checksums, software versions, all tested contrasts, patient-pairing records and model diagnostics to `tables/pca_analysis/`. Keep the prepared bundle with a final repository and rerun this notebook whenever its numerical inputs or analysis settings change. Upstream measurement RDS files are read-only.


In [6]:
capture.output(sessionInfo(), file = file.path(tables_dir, "sessionInfo.txt"))
print(inputs$versions, row.names = FALSE)
print(sessionInfo())


    package version
          R   4.4.0
    missMDA    1.19
 FactoMineR    2.12
R version 4.4.0 (2024-04-24)
Platform: aarch64-apple-darwin20
Running under: macOS 15.8.1

Matrix products: default
BLAS:   /Library/Frameworks/R.framework/Versions/4.4-arm64/Resources/lib/libRblas.0.dylib 
LAPACK: /Library/Frameworks/R.framework/Versions/4.4-arm64/Resources/lib/libRlapack.dylib;  LAPACK version 3.12.0

locale:
[1] C.UTF-8/C.UTF-8/C.UTF-8/C/C.UTF-8/C.UTF-8

time zone: America/Toronto
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

loaded via a namespace (and not attached):
 [1] gtable_0.3.6         shape_1.4.6.1        ggplot2_4.0.0       
 [4] htmlwidgets_1.6.4    ggrepel_0.9.6        lattice_0.22-6      
 [7] numDeriv_2016.8-1.1  vctrs_0.6.5          tools_4.4.0         
[10] Rdpack_2.6.4         generics_0.1.4       parallel_4.4.0      
[13] tibble_3.3.0         cluster_2.1.6        pan_1.9             
[16] pkgc